# CS 340 Project Two: Grazioso Salvare Dashboard

Developed by Matthew Randall

Run the cell below to build and serve the dashboard. The `CRUD_Python_Module.py` file and the Grazioso Salvare logo must be in the same directory as this notebook.

In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import pandas as pd

# CRUD Python module developed in Project One. The dashboard never talks to
# MongoDB directly; every query in this notebook is routed through this class.
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Credentials for the read-only dashboard account created in Project One.
username = "aacuser"
password = "<your aacuser password>"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# The read method returns a list of documents. An empty query document asks
# for every record in the collection, which is the unfiltered starting state
# of the dashboard.
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ returns the '_id' column with an ObjectID type that the
# data_table cannot render, so it is dropped before the frame is displayed.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# Column order is not guaranteed by MongoDB, so the presentation order is
# fixed here. Any column returned by the query that is not listed is appended
# to the end rather than discarded.
PREFERRED_COLUMN_ORDER = [
    'animal_id', 'name', 'animal_type', 'breed', 'color', 'sex_upon_outcome',
    'age_upon_outcome', 'age_upon_outcome_in_weeks', 'date_of_birth',
    'datetime', 'monthyear', 'outcome_type', 'outcome_subtype',
    'location_lat', 'location_long'
]


def order_columns(frame):
    """Return the frame with the preferred columns first, in a fixed order."""
    ordered = [c for c in PREFERRED_COLUMN_ORDER if c in frame.columns]
    remaining = [c for c in frame.columns if c not in ordered]
    return frame[ordered + remaining]


df = order_columns(df)

##################################################
# Rescue Type Queries (Dashboard Specifications)
##################################################
# Each entry is a MongoDB query document built from the Rescue Type and
# Preferred Dog Breeds table in the specifications document. The breed values
# are matched with an anchored regular expression rather than an exact string
# so that mixes are included: '^German Shepherd' matches "German Shepherd",
# "German Shepherd Mix", and "German Shepherd/Labrador Retriever", but it does
# not match unrelated breeds such as "Australian Shepherd Mix". "Doberman
# Pinsch" is used because that is how the breed is spelled in the data set.

WATER_BREEDS = ['Labrador Retriever', 'Chesapeake Bay Retriever', 'Newfoundland']
MOUNTAIN_BREEDS = ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                   'Siberian Husky', 'Rottweiler']
DISASTER_BREEDS = ['Doberman Pinsch', 'German Shepherd', 'Golden Retriever',
                   'Bloodhound', 'Rottweiler']


def breed_pattern(breeds):
    """Build an anchored, case-insensitive regex that also matches mixes."""
    return '^(' + '|'.join(breeds) + ')'


def rescue_query(breeds, sex, min_weeks, max_weeks):
    """Assemble the MongoDB query document for one rescue type."""
    return {
        'animal_type': 'Dog',
        'breed': {'$regex': breed_pattern(breeds), '$options': 'i'},
        'sex_upon_outcome': sex,
        'age_upon_outcome_in_weeks': {'$gte': min_weeks, '$lte': max_weeks}
    }


RESCUE_QUERIES = {
    'water': rescue_query(WATER_BREEDS, 'Intact Female', 26.0, 156.0),
    'mountain': rescue_query(MOUNTAIN_BREEDS, 'Intact Male', 26.0, 156.0),
    'disaster': rescue_query(DISASTER_BREEDS, 'Intact Male', 20.0, 300.0),
    'reset': {}
}

RESCUE_LABELS = {
    'water': 'Water Rescue',
    'mountain': 'Mountain or Wilderness Rescue',
    'disaster': 'Disaster or Individual Tracking',
    'reset': 'All available animals (unfiltered)'
}


def query_dataframe(filter_type):
    """Run the selected rescue query through the CRUD module.

    Returns a DataFrame with the '_id' column removed and the columns in the
    presentation order used by the data table.
    """
    query = RESCUE_QUERIES.get(filter_type, {})
    records = db.read(query)
    dff = pd.DataFrame.from_records(records)

    if dff.empty:
        # Preserve the table structure when a query returns no matches.
        return pd.DataFrame(columns=df.columns)

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return order_columns(dff)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo. The file name differs slightly between the Codio
# workspace and a local checkout, so the first name that exists is used.
LOGO_CANDIDATES = [
    'Grazioso Salvare Logo.png',
    'Grazioso_Salvare_Logo.png',
    'GraziosoSalvareLogo.png'
]

encoded_image = None
for candidate in LOGO_CANDIDATES:
    if os.path.exists(candidate):
        encoded_image = base64.b64encode(open(candidate, 'rb').read())
        break

# The logo is wrapped in an anchor tag pointing at the client home page, as
# required by the branding section of the specifications document.
if encoded_image is not None:
    logo_component = html.A(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            alt='Grazioso Salvare logo',
            style={'height': '160px'}
        ),
        href='https://www.snhu.edu',
        target='_blank'
    )
else:
    logo_component = html.A('Grazioso Salvare', href='https://www.snhu.edu', target='_blank')

app.layout = html.Div([
    html.Center([
        logo_component,
        html.B(html.H1('SNHU CS-340 Dashboard')),
        html.H4('Grazioso Salvare Search-and-Rescue Candidate Finder'),
        # Unique identifier crediting the developer
        html.B('Developed by Matthew Randall')
    ]),
    html.Hr(),

    # Interactive filtering options
    html.Div([
        html.B('Filter by rescue type:'),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset (all animals)', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '25px'},
            inputStyle={'margin-right': '6px'}
        ),
        html.Div(id='filter-summary-id', style={'padding-top': '8px', 'font-style': 'italic'})
    ], style={'padding': '10px 20px'}),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        # Client-friendly table features: one row selectable at a time to drive
        # the map, native sorting and filtering on every column, and pagination
        # so the browser never renders ten thousand rows at once.
        row_selectable='single',
        selected_rows=[0],
        # Column selection drives the cell-highlight callback below
        column_selectable='multi',
        selected_columns=[],
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        page_action='native',
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '110px',
            'maxWidth': '220px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            'fontFamily': 'sans-serif',
            'fontSize': '13px'
        },
        style_header={'backgroundColor': '#B0143C', 'color': 'white', 'fontWeight': 'bold'},
        style_data_conditional=[{'if': {'row_index': 'odd'}, 'backgroundColor': '#F5F5F5'}]
    ),

    html.Br(),
    html.Hr(),

    # This sets up the dashboard so that the pie chart and the geolocation
    # chart are side-by-side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(
                     id='graph-id',
                     className='col s12 m6',
                     style={'width': '50%'}
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                     style={'width': '50%'}
                 )
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'columns'),
     Output('datatable-id', 'selected_rows'),
     Output('datatable-id', 'page_current'),
     Output('datatable-id', 'filter_query'),
     Output('datatable-id', 'sort_by'),
     Output('filter-summary-id', 'children')],
    [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    """Re-query MongoDB through the CRUD module whenever the filter changes.

    Resetting selected_rows and page_current keeps the map and the pagination
    in a valid state when the new result set is smaller than the old one.
    Clearing filter_query and sort_by as well means the Reset option returns
    every widget to its original state, including any column sort or in-table
    filter the user applied by hand.
    """
    dff = query_dataframe(filter_type)

    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in dff.columns]
    data = dff.to_dict('records')
    selected_rows = [0] if len(data) > 0 else []
    summary = '%s: %d matching record(s)' % (RESCUE_LABELS.get(filter_type, ''), len(data))

    return (data, columns, selected_rows, 0, '', [], summary)


# Display the breeds of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    """Draw a pie chart of the breeds currently visible in the data table.

    Only the ten most common breeds are labelled; the remainder are grouped
    into an "Other breeds" slice so the legend stays readable when the table
    is unfiltered.
    """
    if viewData is None or len(viewData) == 0:
        return [html.Div('No records match the selected rescue type.')]

    dff = pd.DataFrame.from_dict(viewData)
    if 'breed' not in dff.columns:
        return [html.Div('No breed data available.')]

    counts = dff['breed'].value_counts()
    if len(counts) > 10:
        top = counts.head(10)
        other = pd.Series({'Other breeds': counts[10:].sum()})
        counts = pd.concat([top, other])

    chart_data = counts.reset_index()
    chart_data.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(
                chart_data,
                names='breed',
                values='count',
                title='Breeds Represented in Current Selection'
            )
        )
    ]


# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    highlighted = [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]

    # Keep the alternating row shading applied in the layout
    return [{'if': {'row_index': 'odd'}, 'backgroundColor': '#F5F5F5'}] + highlighted


# This callback will update the geo-location chart for the selected data entry.
# derived_virtual_data is the set of data available from the datatable in the
# form of a dictionary. derived_virtual_selected_rows is the selected row(s) in
# the table in the form of a list. Only single row selection is permitted, so
# there is only one value in the list. Columns are looked up by name rather
# than by position so the chart does not break if the column order changes.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.Div('No records match the selected rescue type.')]

    dff = pd.DataFrame.from_dict(viewData)

    # Default to the first row when nothing has been selected yet
    if not index:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']

    # Some records have no name recorded in the shelter data
    if pd.isna(name) or str(name).strip() == '':
        name = 'Unnamed'

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            dl.Marker(position=[latitude, longitude], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode. Note: if you have previously
# run a prior app, the default port of 8050 may not be available; if so, try
# setting an alternate port.
app.run_server(debug=False)



 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [13/Aug/2026 15:52:05] "GET /_alive_976ebb35-21d2-43c4-8cab-5a72c20134d0 HTTP/1.1" 200 -


Dash app running on https://elementarmada-zebragentle-3000.codio.io/proxy/8050/


127.0.0.1 - - [13/Aug/2026 15:52:09] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:09] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:09] "GET /_favicon.ico?v=2.8.1 HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:09] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:10] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [13/Aug/2026 15:52:10] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:10] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:10] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:10] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [13/Aug/2026 15:52:11] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:11] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:52:12] "POST /_dash-update-componen